<a href="https://colab.research.google.com/github/krishnanithesh22-stack/Asian-Countries-Plotly-Crime-Rate/blob/main/Asian_Countries_Plotly_Crime_Rate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q plotly pandas lxml requests

import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from io import StringIO

In [5]:
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "}

YEARS = list(range(2021, 2027))

def fetch_year(year):
  url = f"https://www.numbeo.com/crime/rankings_by_country.jsp?title={year}"
  r = requests.get(url, headers = HEADERS, timeout = 30)
  r.raise_for_status()
  table = pd.read_html(StringIO(r.text), attrs = {"id": "t2"})[0]
  table = table[["Country", "Crime Index"]].copy()
  table["Year"] = year
  return table

frames, failed = [], []

for y in YEARS:
  try:
    df_y = fetch_year(y)
    frames.append(df_y)
    print(f"{y}: downloaded {len(df_y)} countries")

  except Exception as e:
    failed.append(y)
    print(f"{y}: FAILED -> {e}")

if frames:
  raw = pd.concat(frames, ignore_index = True)
  print("\nTotal Rows: ", len(raw))
  display(raw.head(20))

else:
  print("No data downloaded.")

2021: downloaded 135 countries
2022: downloaded 142 countries
2023: downloaded 142 countries
2024: downloaded 146 countries
2025: downloaded 147 countries
2026: downloaded 150 countries

Total Rows:  862


,Country,Crime Index,Year
0,Venezuela,84.2,2021
1,Papua New Guinea,80.2,2021
2,South Africa,77.1,2021
3,Afghanistan,76.4,2021
4,Honduras,74.8,2021
5,Trinidad And Tobago,70.9,2021
6,El Salvador,68.8,2021
7,Guyana,68.7,2021
8,Syria,68.1,2021
9,Brazil,67.8,2021


In [6]:
asian_countries = [
    "Afghanistan","Armenia","Azerbaijan","Bahrain","Bangladesh","Bhutan","Brunei",
    "Cambodia","China","Cyprus","Georgia","Hong Kong (China)","India","Indonesia",
    "Iran","Iraq","Israel","Japan","Jordan","Kazakhstan","Kuwait","Kyrgyzstan","Laos",
    "Lebanon","China","Malaysia","Maldives","Mongolia","Myanmar","Nepal",
    "North Korea","Oman","Pakistan","Palestine","Philippines","Qatar","Saudi Arabia",
    "Singapore","South Korea","Sri Lanka","Syria","Taiwan","Tajikistan","Thailand",
    "Timor-Leste","Turkey","Türkiye","Turkmenistan","United Arab Emirates",
    "Uzbekistan","Vietnam","Yemen"
]

asia = raw[raw["Country"].isin(asian_countries)].copy()
asia["Country"] = asia["Country"].replace({"Türkiye": "Turkey"})
asia["Crime Index"] = pd.to_numeric(asia["Crime Index"], errors = "coerce")
asia = asia.dropna(subset = ["Crime Index"])

counts = asia.groupby("Country")["Year"].nunique()
valid = counts[counts >= 3].index
dropped = sorted(set(asian_countries) - set(valid) - {"Türkiye"})
asia = asia[asia["Country"].isin(valid)].sort_values(["Country", "Year"])

print(f"Asian Countries with usable data: {asia['Country'].nunique()}")
print("Not available (no or too little data on Numbeo):")
print(", ".join(dropped))
display(asia.pivot(index="Country", columns="Year", values="Crime Index").round(1))

Asian Countries with usable data: 45
Not available (no or too little data on Numbeo):
Bhutan, Laos, Tajikistan, Timor-Leste, Turkmenistan


Year,2021,2022,2023,2024,2025,2026
Country,,,,,,
Afghanistan,76.4,77.0,78.6,78.3,75.1,74.8
Armenia,23.3,22.1,22.0,22.4,22.1,22.1
Azerbaijan,31.7,32.3,31.8,31.6,31.8,31.6
Bahrain,26.1,24.4,24.0,23.9,24.5,22.3
Bangladesh,64.2,63.4,62.6,61.5,61.6,61.6
Brunei,28.0,29.0,29.0,29.5,29.3,29.4
Cambodia,51.2,50.7,52.9,52.8,51.3,49.7
China,30.2,29.4,26.3,24.9,24.0,23.1
Cyprus,32.0,32.1,31.4,32.4,33.1,33.3


In [7]:
countries = sorted(asia["Country"].unique())
fig = go.Figure()
annotations = {}

for i, c in enumerate(countries):
  d = asia[asia["Country"] == c].sort_values("Year")
  x, y = d["Year"].values, d["Crime Index"].values
  slope = np.polyfit(x, y, 1)[0] if len(x) > 1 else 0
  change = y[-1] - y[0]

  if slope < 0:
    color, verdict = "green", "Becoming SAFER (crime declining)"

  elif slope > 0:
     color, verdict = "red", "Becoming MORE DANGEROUS (crime rising)"

  else:
    color, verdict = "gray", "Stable"

  fig.add_trace(go.Scatter(
      x = x, y = y, mode = "lines+markers+text", name = c,
      text = [f"{v:.1f}" for v in y], textposition = "top center",
      line = dict(color = color, width = 3), marker = dict(size = 9),
      hovertemplate=f"<b>{c}</b><br>Year: %{{x}}<br>Crime Index: %{{y:.1f}}<extra></extra>",
      visible = (i == 0)
  ))

  annotations[c] = dict(
      text = f"<b>{verdict}</b><br>Change {int(x[0])}→{int(x[-1])}: {change:+.1f} points",
      xref = "paper", yref = "paper", x = 0.01, y = 1.12, showarrow = False,
      font = dict(size = 14, color = color), align = "left"
  )

buttons = []

for i, c in enumerate(countries):
  vis = [False] * len(countries)
  vis[i] = True
  buttons.append(dict(
      label=c, method="update",
       args=[{"visible": vis},
             {"title": f"Crime Index Trend: {c} (2021–2026)",
              "annotations": [annotations[c]]}]
  ))

fig.update_layout(
    title = f"Crime Index Trend: {countries[0]} (2021-2026)",
    xaxis = dict(title = "Year", tickmode = "linear", dtick = 1),
    yaxis = dict(title = "Average Crime Index"),
    annotations = [annotations[countries[0]]],
    updatemenus = [dict(buttons=buttons, direction="down", x=0.98, xanchor="right",
                      y=1.15, yanchor="top", showactive=True)],

    template = "plotly_white", height = 600, showlegend = False
)
fig.show()